# Task 1 — Repository Cloning and File Discovery

**Points**: 1.0 / 10  
**Script**: [`src/file_discovery.py`](../../src/file_discovery.py)

---

## 1.1 Approach

We use `git clone --depth=1` (shallow clone) to fetch only the latest commit tree of `huggingface/lerobot`, drastically reducing download size from ~260 MB (full history) to ~70 MB.

After cloning, `src/file_discovery.py` recursively enumerates every `.py` file under the repository tree (excluding `.git/`). Each file is tagged with:
- `rel_path` — path relative to repo root
- `size_bytes` — file size
- `excluded` — boolean flag (True for test files, setup.py, conf.py, etc.)

The full list is persisted to `data/discovered_files.json`, which the Parser Service in Task 2 reads sequentially.

### Why shallow clone?
| Strategy | Download size | History |
|----------|--------------|--------|
| Full clone | ~260 MB | All commits |
| Shallow (`--depth=1`) | ~70 MB | Latest commit only |

For CPG parsing we only need the source files — history is irrelevant.

## 1.2 Cloning the Repository

In [1]:
import subprocess, sys
from pathlib import Path

ROOT     = Path.cwd().parent.parent
REPO_DIR = ROOT / 'repos' / 'lerobot'

if not REPO_DIR.exists():
    result = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/huggingface/lerobot.git', str(REPO_DIR)],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    print('Repository already present — skipping clone.')

r = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'log', '--oneline', '-1'],
    capture_output=True, text=True
)
print('\nHEAD commit:', r.stdout.strip())

Cloning into 'repos/lerobot'...
remote: Enumerating objects: 1284, done.
remote: Counting objects: 100% (1284/1284), done.
remote: Compressing objects: 100% (1152/1152), done.
Filtering content: 100% (50/50), 69.11 MiB | 3.87 MiB/s, done.
Receiving objects: 100% (1284/1284), 70.18 MiB | 4.22 MiB/s, done.

HEAD commit: 93257e3 Merge pull request #4298 from huggingface/remi/push_ds_compat


## 1.3 Running the File Discovery Script

In [2]:
import subprocess, sys
ROOT = Path.cwd().parent.parent
result = subprocess.run(
    [sys.executable, str(ROOT / 'src' / 'file_discovery.py')],
    capture_output=True, text=True, cwd=str(ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

Repository: D:\hk2_2526\big_data\lab04-bigdata\repos\lerobot
Total .py files discovered: 746
  Included (non-test/setup):  543
  Excluded (test/setup/conf): 203
Total size: 8670.0 KB
By top-level directory:
  src                  487
  tests                202
  examples              54
  scripts                2
  setup.py               1

Wrote file list to D:\hk2_2526\big_data\lab04-bigdata\data\discovered_files.json


## 1.4 Analysing the Discovery Results

In [3]:
import json
from pathlib import Path

ROOT = Path.cwd().parent.parent
with open(ROOT / 'data' / 'discovered_files.json', encoding='utf-8') as f:
    disc = json.load(f)

s = disc['summary']

print('╔══════════════════════════════════════════════╗')
print('║       File Discovery Summary — lerobot       ║')
print('╠══════════════════════════════════════════════╣')
print(f"║  Total .py files discovered : {s['total_py_files']:>10}   ║")
print(f"║  Included (to be parsed)    : {s['included_files']:>10}   ║")
print(f"║  Excluded (test/setup/auto) : {s['excluded_files']:>10}   ║")
print(f"║  Total Python source size   : {s['total_size_bytes']//1024:>9,} KB   ║")
print('╚══════════════════════════════════════════════╝')

print('\nDistribution by top-level directory:')
for k, v in s['by_top_level_dir'].items():
    bar = '█' * (v // 10)
    print(f'  {k:<12} {v:4d}  {bar}')

print('\n5 Largest Python files:')
for entry in s['largest_files']:
    print(f"  {entry['size_bytes']/1024:>7.1f} KB  {entry['rel_path']}")

╔══════════════════════════════════════════════╗
║       File Discovery Summary — lerobot       ║
╠══════════════════════════════════════════════╣
║  Total .py files discovered :          746   ║
║  Included (to be parsed)    :          543   ║
║  Excluded (test/setup/auto) :          203   ║
║  Total Python source size   :     8,670 KB   ║
╚══════════════════════════════════════════════╝

Distribution by top-level directory:
  src          487  ████████████████████████████████████████████████
  tests        202  ████████████████████
  examples      54  █████
  scripts        2  
  setup.py       1  

5 Largest Python files:
   188.9 KB  src/lerobot/policies/molmoact2/molmoact2_hf_model/modeling_molmoact2.py
   127.5 KB  src/lerobot/policies/wall_x/qwen_model/qwen2_5_vl_moe.py
   121.1 KB  src/lerobot/policies/xvla/modeling_florence2.py
   109.8 KB  src/lerobot/policies/groot/processor_groot.py
    88.6 KB  src/lerobot/policies/wall_x/modeling_wall_x.py


## 1.5 Exclusion Logic

Files are excluded if any of the following conditions hold:

```python
EXCLUDE_NAME_PATTERNS = ('setup.py', 'conf.py')
EXCLUDE_DIR_PARTS     = ('tests', 'test')

def is_excluded(rel_path: Path) -> bool:
    name = rel_path.name
    if name in EXCLUDE_NAME_PATTERNS or name.startswith('test_') or name.endswith('_test.py'):
        return True
    if any(part in EXCLUDE_DIR_PARTS for part in rel_path.parts[:-1]):
        return True
    return False
```

This removes 203 test/setup files, leaving **543 files** for CPG parsing.

In [4]:
included = [r for r in disc['files'] if not r['excluded']][:5]
excluded = [r for r in disc['files'] if r['excluded']][:5]

print('Sample included files (first 5):')
for r in included:
    print(f"  {r['rel_path']:<50s} ({r['size_bytes']:>7,} B)")

print('\nSample excluded files (first 5):')
for r in excluded:
    reason = '[test dir]' if 'test' in r['rel_path'].lower() else '[name match]'
    print(f"  {r['rel_path']:<50s} ({r['size_bytes']:>7,} B)  {reason}")

Sample included files (first 5):
  examples/annotations/run_hf_job.py             (  2,929 B)
  examples/backward_compatibility/replay.py       (  3,106 B)
  examples/dataset/create_progress_videos.py      ( 23,930 B)
  examples/dataset/load_lerobot_dataset.py        (  6,477 B)
  examples/dataset/use_dataset_image_transforms.py(  6,985 B)

Sample excluded files (first 5):
  tests/cameras/test_cameras.py                   (  4,502 B)  [test dir]
  tests/conftest.py                               (  2,234 B)  [test dir]
  tests/datasets/test_dataset_tools.py            ( 55,167 B)  [test dir]
  tests/datasets/test_datasets.py                 ( 74,285 B)  [test dir]
  tests/policies/test_policies.py                 ( 14,021 B)  [test dir]


## 1.6 Reflection

### ✅ What worked
- `--depth=1` shallow clone completed in ~35 seconds, downloading only 70 MB instead of the full ~260 MB repository.
- `pathlib.rglob('*.py')` correctly found all 746 Python files, including nested policy model files in deep sub-directories.
- The exclusion logic cleanly separated 203 test/setup files, leaving 543 production source files for CPG parsing.
- The `discovered_files.json` manifest format (`rel_path`, `size_bytes`, `excluded`) provides a clean contract between Task 1 and Task 2.

### ⚠️ Issues encountered
- Several large policy model files exceeded 100 KB (largest: `modeling_molmoact2.py` at 188.9 KB). These generate very large AST trees, which were the slowest files to parse in Task 2.
- LeRobot uses both `tests/` and `test_*.py` naming conventions, requiring both exclusion rules.

### 💡 Resolution
- Implemented `--limit N` flag in the Parser Service so we could smoke-test with a small subset before running the full 543-file pipeline.
- Stable IDs (SHA-256 hash of file path + position) ensure that if any large file is reprocessed, no duplicates are created downstream.